# Push Container Image to Azure Container Registry

This notebook builds and publishes the optimized SLM inference container to Azure Container Registry for deployment.

## What This Notebook Does

1. **Builds Container Image**: Creates Docker image with optimized model and inference server
2. **Multi-Architecture Support**: Builds for both AMD64 (x86_64) and ARM64 platforms
3. **Tags and Pushes**: Uploads image to ACR with semantic versioning
4. **Vulnerability Scanning**: Optionally scans for security issues before deployment
5. **Validates Push**: Confirms image is available in registry and manifests are correct

## Why Use Azure Container Registry?

**Centralized Management:**
- Single source of truth for container images
- Version control with tags (v1.0, v1.1, latest)
- Integration with Azure security and compliance tools

**Performance:**
- Geo-replicated storage for fast pulls worldwide
- Network-optimized transfers within Azure
- Cached layers reduce rebuild times

**Security:**
- Private registry (not publicly accessible)
- Azure AD integration for authentication
- Image scanning and vulnerability detection
- Content trust and image signing

## Multi-Architecture Builds

**Why support multiple architectures?**

Many embedded devices use ARM64 processors (Raspberry Pi, Jetson Nano, AWS Graviton):
- **AMD64 (x86_64)**: Desktop, cloud VMs, Intel/AMD servers
- **ARM64 (aarch64)**: Embedded devices, edge compute, mobile chips

Docker Buildx enables building both architectures from a single Dockerfile using QEMU emulation.

## Container Size Optimization

The Dockerfile uses multi-stage builds to minimize image size:

```
Base Image:          ~1.5GB (Python + system libraries)
+ Model (quantized): ~500MB (int8 quantization)
+ Dependencies:      ~200MB (PyTorch CPU, FastAPI, uvicorn)
+ Inference Code:    ~5MB (Python scripts)
─────────────────────────────────
Final Image:         ~2.2GB (or <500MB with aggressive optimization)
```

**Size reduction techniques:**
- Use slim base images (python:3.11-slim)
- Remove build dependencies after compilation
- Delete pip cache and temporary files
- Use quantized models (int8/int4)
- Consider distroless or Alpine base images

## Tagging Strategy

**Semantic Versioning:**
```
<registry>.azurecr.io/slm-inference:v1.0.0  # Specific version
<registry>.azurecr.io/slm-inference:v1.0    # Minor version
<registry>.azurecr.io/slm-inference:v1      # Major version
<registry>.azurecr.io/slm-inference:latest  # Latest stable
<registry>.azurecr.io/slm-inference:dev     # Development/testing
```

**Best Practice:** Always use specific version tags in production, not `latest`.

## Security Scanning

Optional vulnerability scanning with Trivy:
- Detects CVEs in base image and dependencies
- Identifies outdated packages with known exploits
- Generates SBOM (Software Bill of Materials)
- Gates deployment based on severity threshold

## Prerequisites

- Completed notebook `08-optimize-model.ipynb`
- Docker installed and running
- Azure CLI authenticated (`az login`)
- ACR created (via notebook `01-setup-infrastructure.ipynb`)
- Sufficient disk space (~10GB) for build cache

## Expected Duration

- Single arch build: ~5-10 minutes
- Multi-arch build: ~15-30 minutes (QEMU emulation slower)
- Push to ACR: ~5-10 minutes depending on network

## Cost Considerations

**ACR Storage:**
- Basic SKU: $0.167/day (~$5/month) + storage costs
- Standard SKU: $0.667/day (~$20/month) + storage, geo-replication
- Each image version consumes storage (layers are deduplicated)

# Notebook 11: Push Image to Azure Container Registry (ACR)



This notebook demonstrates best-practice workflows for building, tagging, multi-arch publishing, verifying, and (optionally) scanning a container image for the SLM inference server.



## 0. Prerequisites

- Azure CLI installed (`az --version`)

- Logged in: `az login`

- ACR created: `az acr create -n <registry_name> -g <resource_group> --sku Basic`

- Docker Buildx plugin available (`docker buildx ls`)

- Optional security scanner (e.g. `trivy`) installed

- Environment variables or parameters for registry, image name, version



> Principle: Never hardcode credentials. ACR auth uses your Azure CLI context or Managed Identity in CI.



## 1. Parameters



Run the next cell to define local variables (adjust as needed).




## 2. Build Single-Architecture Image

Use a local build first for validation. Then tag and push.



## 3. Multi-Architecture Buildx

Create an isolated builder instance and build for `linux/amd64,linux/arm64`.



## 4. Vulnerability Scanning (Optional)

Run a scan tool before pushing or after to assess the pushed image.



## 5. Verification & Cleanup

Check tags, manifests, and optionally remove local images to save space.



---

Proceed with the cells below.